In [79]:
import numpy as np
import os,sys
sys.path.append(os.path.realpath('../lib'))
from graph_multimodalsentence import  graph_colors,draw_rel,draw_transp,Graph,wl_labeling,find_thresh,sp_to_adjency
from ot_distances import Fused_Gromov_Wasserstein_distance
from sentence_transformers import SentenceTransformer
from FGW import fgw_barycenters
import seaborn as sns

In [80]:
method='shortest_path_weight'
embedding = SentenceTransformer('sentence-transformers/bert-base-nli-mean-tokens')
def create_graph_from_data(x,embedding):
    graphs=Graph()
    graphs.add_sentence(x,embedding)
    return graphs

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4172.21it/s]


In [81]:
import pandas as pd
data = pd.read_csv('../data_with_text_tag\generated_instructions_with_intention_explanation_full_benevolence_gender_neutral_mistral_qa.csv')
data_high = data[data['benevolence'] == 'High']
data_low = data[data['benevolence'] == 'Low']
data_medium = data[data['benevolence'] == 'Medium']
graphs_low=[create_graph_from_data(x,embedding) for x in data_low.instruction_with_intention]
graphs_medium=[create_graph_from_data(x,embedding) for x in data_medium.instruction_with_intention]
graphs_high=[create_graph_from_data(x,embedding) for x in data_high.instruction_with_intention]


Look Nails
Dismissing Gesture
Cocky Head Turn (Right)
Look Nails
Dismissing Gesture
Shrugging
Look Nails
Dismissing Gesture
Look Nails
Dismissing Gesture
Look Nails
Dismissing Gesture
Look Nails
Dismissing Gesture
Look Nails
Dismissing Gesture
Look Nails
Dismissing Gesture
Neutral Idle
Look Nails
Dismissing Gesture
Shrugging
Look Nails
Dismissing Gesture
Neutral Idle
Look Nails
Talking 3
Dismissing Gesture
Look Nails
Pointing
Dismissing Gesture
Look Nails
Pointing
Dismissing Gesture
Look Nails
Dismissing Gesture
Cocky Head Turn (Right)
Look Nails
Dismissing Gesture
Neutral Idle
Look Nails
Dismissing Gesture
Shrugging
Look Nails
Talking 2
Dismissing Gesture
Look Nails
Dismissing Gesture
Shrugging
Look Nails
Talking 3
Dismissing Gesture
Look Nails
Talking 3
Dismissing Gesture
Look Nails
Dismissing Gesture
Shrugging
Look Nails
Dismissing Gesture
Cocky Head Turn (Right)
Look Nails
Dismissing Gesture
Neutral Idle
Look Nails
Dismissing Gesture
Neutral Idle
Look Nails
Dismissing Gesture
Look 

In [82]:
def createbary(graphs):
    for x in graphs:
        #if x.nx_graph has no nodes of edges, remove it from the list
        if len(x.nodes())==0 or len(x.edges())==0:
            graphs.remove(x)
    Cs=[x.distance_matrix(force_recompute=True,method=method) for x in graphs]
    ps=[np.ones(len(x.nodes()))/len(x.nodes()) for x in graphs]
    Ys=[x.values() for x in graphs]
    lambdas=np.array([np.ones(len(Ys))/len(Ys)]).ravel()
    sizebary=(15,768) # we choose a barycenter with 15 nodes
    init_X=np.zeros((sizebary[0],sizebary[1]))
    D1_high,C1_high,log=fgw_barycenters(sizebary[0],Ys,Cs,ps,lambdas,alpha=0.14,init_X=init_X)
    bary_high = Graph()
    from_numpy_array(bary_high, sp_to_adjency(C1_high, threshinf=0, threshsup=find_thresh(C1_high, sup=100, step=100)[0]))
    for i in range(len(D1_high)):
        bary_high.nx_graph.add_node(i, attr_name=D1_high[i])
    return bary_high

In [83]:
def from_numpy_array(G, adj_matrix, source_node_offset=0):
        """
        Populates the graph from a numpy adjacency matrix.
        
        Args:
            adj_matrix (np.ndarray): Square matrix where adj_matrix[i, j] is the weight.
            source_node_offset (int): The starting ID for nodes. 
                                      Useful if you are adding this matrix to an existing graph.
        """
        import numpy as np
        
        rows, cols = adj_matrix.shape
        if rows != cols:
            raise ValueError("Adjacency matrix must be square.")

        # 1. Ensure all nodes exist in the graph first
        for i in range(rows):
            node_id = i + source_node_offset
            if not G.nx_graph.has_node(node_id):
                G.nx_graph.add_node(node_id)
        
        # 2. Iterate through the matrix to add edges
        # We use np.nonzero to avoid iterating over thousands of zeros in sparse matrices
        nonzero_indices = np.nonzero(adj_matrix)
        
        for i, j in zip(*nonzero_indices):
            weight = adj_matrix[i, j]
            u = i + source_node_offset
            v = j + source_node_offset
            
            # Add edge with the weight from the matrix
            # You can add additional default attributes here (e.g., type='matrix_imported')
            G.nx_graph.add_edge(u, v, weight=weight)
            
        # 3. Update the internal node counter to prevent ID collisions in future additions
        G.n_nodes = max(G.n_nodes, rows + source_node_offset)
        
        print(f"Successfully imported {len(nonzero_indices[0])} edges from matrix.")


In [84]:
def intracategorieDivergence(graphs):
    bary=createbary(graphs)
    bary_distance=[Fused_Gromov_Wasserstein_distance(alpha=0.14,features_metric='sqeuclidean',method=method).graph_d(bary,x) for x in graphs]
    return bary_distance

In [85]:
intracategorieDivergence_high=intracategorieDivergence(graphs_high)
intracategorieDivergence_low=intracategorieDivergence(graphs_low)
intracategorieDivergence_medium=intracategorieDivergence(graphs_medium)

Successfully imported 210 edges from matrix.
Successfully imported 210 edges from matrix.
Successfully imported 210 edges from matrix.


In [86]:
print("Intracategory Divergence for High Ability:", np.mean(intracategorieDivergence_high), "with std:", np.std(intracategorieDivergence_high))
print("Intracategory Divergence for Low Ability:", np.mean(intracategorieDivergence_low), "with std:", np.std(intracategorieDivergence_low))
print("Intracategory Divergence for Medium Ability:", np.mean(intracategorieDivergence_medium), "with std:", np.std(intracategorieDivergence_medium))

Intracategory Divergence for High Ability: 0.18128362181938712 with std: 0.020355965371410222
Intracategory Divergence for Low Ability: 0.16115066258925997 with std: 0.02742481584796608
Intracategory Divergence for Medium Ability: 0.19062938403506208 with std: 0.026675187662483793
